# Tests: can the coefficients be improved by small changes?

We have used the genetic algorithm to find 100 different combinations of the 25 coefficients. The way that the algorithm works means that it is exceedingly unlikely that the true best answer has been found. It's quite likely that each set of answers is close to an even better answer. To improve on our current results, we can take each set of 25 coefficients and nudge each value, making it either a bit larger or a bit smaller, and find a number of small nudges that vastly improve the result.

In this notebook we test how good the coefficients are by using:
+ "sum of square residuals" - a test of how close all of the separate MSOA-level admissions are to the observed values. MSOA-level data is available for five deprivation quantiles. Lower values are better fits.
+ "wrongness ratio" - a test of how close the England-level admissions in each age band are to the observed values. Lower values are better fits, with the lowest possible value being 1 (perfect fit).
+ "fitness" - the two previous measures multiplied together. Lower values are better.

We nudge the values by either adding or subtracting a bit. For example, a coefficient of 0.003 can be nudged up one significant figure to 0.004, or down one to 0.002. We could also move it two clicks to 0.005 or 0.001.

_The problem:_ when each coefficient has five options (stay the same, nudge up one, up two, down one, or down two), and there are 25 coefficients, then the total number of combinations is 5^25 = 3x10^17, three hundred quadrillion or three hundred million billion possible combinations. For context, the Sun is around 1.5 hundred quadrillion seconds old so if we'd tried two combinations a second since the Sun formed then we'd only just now have finished. This is far too many possibilities to check everything!

It's easier to check every combination for just the five coefficients that belong to one age band or to one deprivation level. There are five coefficients and 5^5=3125 combinations.

In this notebook we test whether there are any better combinations to be found by trying a few nudges at random. Then we try every combination of nudges for each deprivation level _or_ each age band for the best set of coefficients.

We perform four tests to see if the overall fit can be improved:
1. Manually cherry-pick coefficients,
2. Try some offsets at random,
3. Optimise just the deprivation level coefficients, and
4. Optimise just the age band coefficients.

__Inputs__

+ MSOA-level admission numbers, numbers of people in each age band.
+ Probability of stroke given age band, i.e. SSNAP-derived coefficients
+ Total number of admissions in each age band from SSNAP
+ 100 sets of "best" coefficients from 100 runs of the genetic algorithm

__Method__

1. Can the starting coefficients be improved by manually nudging the values?
    + _Method:_ Take the best set of coefficients. Look in the other sets' results for each deprivation level separately. Manually alter the best set of coefficients to match the best results for each deprivation level separately. 
    + _Result:_ mixed. We find better sum of square residuals but worse overall fitness.
2. Can both sum of square residuals and fitness be improved?
    + _Method:_ Take the best set of coefficients. Randomly add or subtract one step from a few of the coefficients and recalculate the fitness. Repeat for up to 10,000 randomly-generated shifts.
    + _Result:_ Success. Both sum of square residuals and fitness can be improved at the same time.
   
From this test, we know that there are better answers to be found than the starting values. It's then a case of making sure we've checked everything reasonable and haven't missed any excellent answers. The previous test tried some shifts at random but this doesn't guarantee that we checked every good answer or even most good answers.

To try everything would take too many combinations, so instead try:

3. Combine the best results from each of the best five deprivation levels. Does fitness improve?
    + _Method:_ For each deprivation level, take the coefficients from the best set. Find every variation of this best set when adding or subtracting one or two steps or not changing each of the five coefficients. Calculate sum of square residuals for each new variation separately. Gather the best results for each of the deprivation levels into a new set of 25 coefficients.
    + _Result:_ Failure. The sum of square residuals improved, but fitness worsens.
4. Combine the best results from each of the best five age bands. Does fitness improve?
    + _Method:_ For each age band, take the coefficients from the best set. Find every variation of this best set when adding or subtracting one or two steps or not changing each of the five coefficients. Calculate wrongness ratio for each new variation separately. Gather the best results for each of the deprivation levels into a new set of 25 coefficients.
    + _Result:_ Failure. The sum of square residuals improved, but fitness worsens.

These tests show that we won't be able to improve the overall fitness by looking at either deprivation level or age band separately. All 25 coefficients will have to be considered together to find improvements.

## Code setup

In [1]:
import polars as pl
import os
import numpy as np
import itertools

## Load data

### Admissions

MSOA-level admissions data and numbers of people in each age band:

In [2]:
path_to_msoa_stats = os.path.join('data', 'msoa_cleaned.csv')

df_stats = pl.read_csv(path_to_msoa_stats)

In [3]:
df_stats.head()

MSOA,admissions,IMD2019Score,All persons,country,good_health,fair health,bad health,prop_good_health,prop_fair health,prop_bad health,MSOA11CD,age_65_proportion,age_70_proportion,age_75_proportion,age_less65_proportion,age_over80_proportion,total_health,age_less65,age_65,age_70,age_75,age_over80,depriv_quantile_min,depriv_quantile_max
str,f64,f64,i64,str,i64,i64,i64,f64,f64,f64,str,f64,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64
"""Adur 001""",14.333333,16.924833,8815,"""E""",6799,1251,474,0.79763,0.146762,0.055608,"""E02006534""",0.0559,0.0528,0.0422,0.7872,0.062,8524,6939.168,492.7585,465.432,371.993,546.53,0.4,0.6
"""Adur 002""",7.333333,6.4704,7263,"""E""",5537,838,259,0.83464,0.126319,0.039041,"""E02006535""",0.0578,0.0774,0.0492,0.7467,0.0692,6634,5423.2821,419.8014,562.1562,357.3396,502.5996,0.8,1.0
"""Adur 003""",9.333333,13.7334,7354,"""E""",5820,969,311,0.819718,0.136479,0.043803,"""E02006536""",0.0609,0.0582,0.0421,0.7729,0.0661,7100,5683.9066,447.8586,428.0028,309.6034,486.0994,0.6,0.8
"""Adur 004""",21.0,26.199857,10582,"""E""",7872,1546,709,0.777328,0.152661,0.070011,"""E02006537""",0.0465,0.0438,0.0367,0.8091,0.0638,10127,8561.8962,492.063,463.4916,388.3594,675.1316,0.2,0.4
"""Adur 005""",13.666667,11.7948,9059,"""E""",7106,1081,339,0.833451,0.126789,0.039761,"""E02006538""",0.0597,0.067,0.0425,0.7643,0.0662,8526,6923.7937,540.8223,606.953,385.0075,599.7058,0.6,0.8


Pick out column names for the health and age proportions:

In [4]:
props_age = [
    'age_less65_proportion', 'age_65_proportion', 'age_70_proportion',
    'age_75_proportion', 'age_over80_proportion'
]
age_numbers = [p.replace('_proportion', '') for p in props_age]
qmin_list = sorted(df_stats['depriv_quantile_min'].unique())

# Names of coeffs:
coeff_names = [f'{a}_q{str(round(q, 1)).replace(".", "")}'
               for q in qmin_list for a in age_numbers]
# Check the first few:
print(coeff_names[:3])

# Quantile names:
quantile_str_list = sorted(list(set([c.split('_')[-1] for c in coeff_names])))

['age_less65_q00', 'age_65_q00', 'age_70_q00']


Gather admissions data by deprivation quantile:

In [5]:
admissions_lists = []
x_lists = []

for qmin in qmin_list:
    df_stats_here = df_stats.filter(df_stats['depriv_quantile_min'] == qmin)
    # MSOA data in the same order as those coefficients:
    x_lists_here = [df_stats_here[a] for a in age_numbers]
    admissions_here = df_stats_here['admissions'].to_numpy().tolist()
    # Store:
    admissions_lists.append(admissions_here)
    x_lists.append(x_lists_here)

### Age-admissions coefficients

Starting SSNAP coefficients:

In [6]:
df_pop_admissions = pl.read_csv(os.path.join('outputs', 'ssnap_coeffs.csv'))

In [7]:
df_pop_admissions

Age Groups,population,prop_of_all_pop,count,prop_of_all_admissions,admissions_annual,admissions_annual_boost,prob_stroke_given_age
str,i64,f64,i64,f64,f64,f64,f64
"""Under 65""",45933245,0.816055,38827,0.231449,12942.33333,18737.66819,0.000408
"""65-69""",2796740,0.049687,15324,0.091347,5108.0,7395.26689,0.002644
"""70-74""",2779326,0.049378,21508,0.12821,7169.33333,10379.62674,0.003735
"""75-79""",1940686,0.034478,24150,0.143959,8050.0,11654.63948,0.006005
"""80 and over""",2836964,0.050402,67947,0.405035,22649.0,32790.7987,0.011558


Pick out SSNAP coefficients:

In [8]:
coeffs_ssnap = df_pop_admissions['prob_stroke_given_age'].to_numpy()

coeffs_ssnap

array([0.000408, 0.002644, 0.003735, 0.006005, 0.011558])

Turn them into a dictionary:

In [9]:
labels = ['less65', '65', '70', '75', 'over80']
coeffs_ssnap_dict = (
    dict(zip(labels, df_pop_admissions['prob_stroke_given_age'].to_numpy())))

coeffs_ssnap_dict

{'less65': 0.000408,
 '65': 0.002644,
 '70': 0.003735,
 '75': 0.006005,
 'over80': 0.011558}

Pick out admissions numbers:

In [10]:
admissions_by_age = (
    df_pop_admissions['admissions_annual_boost'].to_numpy().flatten())

### Best results from genetic algorithm

Data stored as scale factors for the SSNAP-derived coefficients.

In [11]:
df_best_gens = pl.read_csv(os.path.join('outputs', 'best_inds_deap.csv'))

In [12]:
df_best_gens.head()

dir,# gen,age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,age_less65_q02,age_65_q02,age_70_q02,age_75_q02,age_over80_q02,age_less65_q04,age_65_q04,age_70_q04,age_75_q04,age_over80_q04,age_less65_q06,age_65_q06,age_70_q06,age_75_q06,age_over80_q06,age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08,fitness,r2_all,r2_q00,r2_q02,r2_q04,r2_q06,r2_q08
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""randomseed00""",33.0,1.3,1.2,1.332,1.237,1.2,1.026,1.145,1.151,1.041,1.1,1.0,1.0,0.988,0.938,1.0,0.862,0.9,0.945,0.938,0.937,0.8,0.9,0.732,0.832,0.874,238.372,0.584365,0.502719,0.579147,0.616005,0.598245,0.606993
"""randomseed01""",33.0,1.1,1.3,1.305,1.3,1.231,1.1,1.1,1.1,1.023,1.1,1.0,1.0,1.0,1.0,1.0,0.969,0.9,0.9,0.892,0.9,0.837,0.849,0.8,0.8,0.894,238.507,0.575882,0.471584,0.579147,0.616005,0.583824,0.606993
"""randomseed02""",43.0,1.273,1.308,1.175,1.2,1.214,0.981,1.1,1.103,1.1,1.131,0.949,0.987,1.046,1.0,1.0,0.949,0.979,0.896,0.9,0.9,0.844,0.773,0.861,0.8,0.88,238.037,0.58383,0.485431,0.583466,0.616005,0.606683,0.606993
"""randomseed03""",28.0,1.171,1.3,1.327,1.186,1.3,1.1,1.198,1.1,1.1,1.1,1.042,1.0,1.0,1.0,0.971,0.881,0.9,0.894,1.0,0.9,0.8,0.771,0.808,0.7,0.9,239.49,0.585993,0.508469,0.583466,0.631482,0.605116,0.580367
"""randomseed04""",39.0,1.2,1.2,1.25,1.225,1.293,1.1,1.1,1.11,1.069,1.1,0.976,1.019,1.0,1.0,0.961,0.925,0.904,0.9,0.915,0.91,0.8,0.9,0.844,0.8,0.879,238.453,0.591769,0.508469,0.579147,0.631482,0.612783,0.606993


Convert the scale factors to the actual age-deprivation coefficient values using the SSNAP coefficients.

In [13]:
for coeff in coeff_names:
    key = coeff.split('_')[1]
    new_data = df_best_gens[coeff] * coeffs_ssnap_dict[key]
    
    df_best_gens = df_best_gens.with_columns(pl.Series(coeff, new_data))

Round results:

In [14]:
# Set up dictionary with how many decimal places to round to:
labels = ['less65', '65', '70', '75', 'over80']
# round_dict = dict(zip(labels, [4, 3, 3, 3, 3]))
# round_dict = dict(zip(labels, [5, 4, 4, 4, 3]))
round_dict = dict(zip(labels, [5, 4, 4, 4, 4]))

for coeff in coeff_names:
    key = coeff.split('_')[1]
    new_data = np.round(df_best_gens[coeff], round_dict[key])
    
    df_best_gens = df_best_gens.with_columns(pl.Series(coeff, new_data))

Drop the measures of goodness of fit now that the coefficients have been rounded:

In [15]:
df_best_gens = df_best_gens.drop(
    ['fitness'] + [c for c in df_best_gens.columns if c.startswith('r2')])

## Recalculate fitness

Use similar functions to previous notebooks to calculate sum of square residuals and fitness.

In [16]:
def predict_admissions(x_lists, coeffs):
    """
    x_lists: np.array.
    coeffs: np.array.
    
    Have to have same number of coeffs as x_lists.
    """
    # Predicted admissions:
    # yhat = (x_lists * coeffs.reshape(len(coeffs), 1)).sum(axis=0)
    yhat = sum(
        [x_lists[i] * coeffs[i] for i in range(len(coeffs))]
    )
    return yhat.to_numpy().tolist()

In [17]:
def predict_admissions_each_age(x_lists, coeffs):
    """
    x_lists: np.array.
    coeffs: np.array.
    
    Have to have same number of coeffs as x_lists.
    """
    # Predicted admissions:
    yhat = (
        [(x_lists[i] * coeffs[i]).sum() for i in range(len(coeffs))]
    )
    return yhat

Goodness check option 1: This calculates the sum of the square of the differences between predicted and actual admission numbers:

In [18]:
def find_square_residuals(yhat, y):
    # Difference from actual:
    sqres = (np.array(yhat) - np.array(y))**2.0
    # Sum of differences:
    sum_sqres = sqres.sum()
    return np.sqrt(sum_sqres)

In [19]:
def many_sum_sqres(individual, x_lists, admissions_lists):
    # Predictions for each MSOA:
    predictions_lists = []
    sum_sqres_by_depriv = []
    for i in range(5):
        coeffs = individual[(i*5):(i*5)+5]  # coeffs for this depriv.
        yhat = predict_admissions(x_lists[i], coeffs)
        sum_sqres = find_square_residuals(yhat, admissions_lists[i])
        predictions_lists.append(yhat)
        sum_sqres_by_depriv.append(sum_sqres)


    # Combine lists:
    observed_all = sum(admissions_lists, [])
    predicted_all = sum(predictions_lists, [])
    # Calculate sum of square residuals across all MSOA:
    sum_sqres = find_square_residuals(predicted_all, observed_all)

    return sum_sqres, sum_sqres_by_depriv

In [20]:
# the goal ('fitness') function to be maximized
def eval_admissions(individual, admissions_lists, x_lists, admissions_by_age):

    # Predictions for each age band across England:
    predictions_by_age = [0.0] * 5
    for i in range(5):
        # Population numbers for areas in this quantile:
        x_lists_here = x_lists[i]
        # Separate prediction for each deprivation quantile:
        coeffs = individual[(i*5):(i*5)+5] #* np.array(coeffs_ssnap)
        yhat_list = predict_admissions_each_age(x_lists_here, coeffs)
        # predictions_lists.append(yhat)
        for j, y in enumerate(yhat_list):
            predictions_by_age[j] += y
    # Divide the admissions by age band by the observed values:
    for j in range(len(predictions_by_age)):
        predictions_by_age[j] /= admissions_by_age[j]
    # Wrongness ratio factor:
    rat = sum(np.abs(np.array(predictions_by_age) - 1.0)) + 1.0

    sum_sqres, sum_sqres_by_depriv = many_sum_sqres(individual, x_lists, admissions_lists)
    
    return (sum_sqres, rat, sum_sqres * rat, predictions_by_age, sum_sqres_by_depriv)

In [21]:
def eval_fitness_to_df(df_best_gens, admissions_lists, x_lists, admissions_by_age, coeff_names, concat=False):
    eval_dict = {}
    eval_labels = [
        'sum_sqres',
        'wrong_rat',
        'fitness',
        'wrong_rat_under65',
        'wrong_rat_65',
        'wrong_rat_70',
        'wrong_rat_75',
        'wrong_rat_over80',
        'sum_sqres_q00',
        'sum_sqres_q02',
        'sum_sqres_q04',
        'sum_sqres_q06',
        'sum_sqres_q08',
    ]
    eval_dict = dict(zip(eval_labels, [[] for e in eval_labels]))
    
    for d in range(len(df_best_gens)):
        df = df_best_gens[d]
        # Pick out coefficients:
        coeffs = df[coeff_names].to_numpy().flatten()
        sum_sqres, rat, fitness, ratios_by_age, sum_sqres_by_depriv = eval_admissions(
            coeffs,
            admissions_lists,
            x_lists,
            admissions_by_age
        )
        eval_dict['sum_sqres'].append(sum_sqres)
        eval_dict['wrong_rat'].append(rat)
        eval_dict['fitness'].append(fitness)
        eval_dict['wrong_rat_under65'].append(ratios_by_age[0])
        eval_dict['wrong_rat_65'].append(ratios_by_age[1])
        eval_dict['wrong_rat_70'].append(ratios_by_age[2])
        eval_dict['wrong_rat_75'].append(ratios_by_age[3])
        eval_dict['wrong_rat_over80'].append(ratios_by_age[4])
        eval_dict['sum_sqres_q00'].append(sum_sqres_by_depriv[0])
        eval_dict['sum_sqres_q02'].append(sum_sqres_by_depriv[1])
        eval_dict['sum_sqres_q04'].append(sum_sqres_by_depriv[2])
        eval_dict['sum_sqres_q06'].append(sum_sqres_by_depriv[3])
        eval_dict['sum_sqres_q08'].append(sum_sqres_by_depriv[4])

    # Either start a new blank dataframe or add the fitnesses to the
    # input df:
    df_new = df_best_gens if concat else pl.DataFrame()
    for key, v in eval_dict.items():
        df_new = df_new.with_columns(pl.Series(key, v))
    return df_new

Recalculate fitnesses for the genetic algorithm outputs:

In [22]:
df_best_gens = eval_fitness_to_df(df_best_gens, admissions_lists, x_lists, admissions_by_age, coeff_names, concat=True)

View results:

In [23]:
df_best_gens.sort('fitness').head()

dir,# gen,age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,age_less65_q02,age_65_q02,age_70_q02,age_75_q02,age_over80_q02,age_less65_q04,age_65_q04,age_70_q04,age_75_q04,age_over80_q04,age_less65_q06,age_65_q06,age_70_q06,age_75_q06,age_over80_q06,age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08,sum_sqres,wrong_rat,fitness,wrong_rat_under65,wrong_rat_65,wrong_rat_70,wrong_rat_75,wrong_rat_over80,sum_sqres_q00,sum_sqres_q02,sum_sqres_q04,sum_sqres_q06,sum_sqres_q08
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""randomseed77""",27.0,0.00049,0.0034,0.0052,0.0076,0.014,0.00045,0.0029,0.0041,0.0066,0.013,0.0004,0.0026,0.0035,0.0057,0.011,0.00035,0.0024,0.0034,0.0054,0.011,0.00035,0.0023,0.003,0.0048,0.01,237.338713,1.006701,238.929181,1.001115,0.999649,0.998994,0.999344,0.996426,110.556611,108.388027,106.9721,103.167012,101.353211
"""randomseed86""",39.0,0.00048,0.0034,0.0042,0.0072,0.016,0.00045,0.0029,0.0041,0.0066,0.013,0.00041,0.0025,0.0037,0.006,0.011,0.00036,0.0024,0.0035,0.0054,0.011,0.00034,0.0024,0.0034,0.0048,0.009,238.120576,1.003774,239.019253,1.000612,0.999867,1.002302,1.000544,1.000183,110.617949,108.388027,106.584283,103.270172,103.402979
"""randomseed28""",19.0,0.00049,0.0034,0.0049,0.0078,0.013,0.00045,0.0031,0.0041,0.0066,0.013,0.0004,0.0026,0.0037,0.0057,0.012,0.00037,0.0024,0.0034,0.0053,0.011,0.00033,0.0021,0.003,0.0048,0.01,237.957746,1.007609,239.768263,1.00109,0.997204,0.998657,1.000174,1.002206,111.694602,108.604283,106.434009,103.311688,101.743852
"""randomseed40""",70.0,0.00052,0.0034,0.005,0.0068,0.013,0.00045,0.0029,0.0041,0.0066,0.013,0.00037,0.0027,0.0037,0.006,0.011,0.00036,0.0026,0.0034,0.0056,0.011,0.00033,0.002,0.003,0.0048,0.011,237.330185,1.010803,239.894083,0.998015,0.999889,1.002593,0.998885,1.004999,111.686058,108.388027,106.37623,103.493876,100.384223
"""randomseed34""",33.0,0.00049,0.0038,0.0045,0.0074,0.014,0.00045,0.0029,0.004,0.0062,0.013,0.00038,0.0026,0.0037,0.0058,0.012,0.00037,0.0024,0.0034,0.0055,0.01,0.00035,0.002,0.0033,0.0051,0.01,237.425568,1.011679,240.198419,1.000598,0.999954,0.996891,1.00242,0.994494,110.731876,108.190202,105.895539,104.698947,101.136034


Pick out the best coeffs so far:

In [24]:
df_coeffs_best_deap = df_best_gens.sort('fitness')[0]
coeffs_best_deap = df_coeffs_best_deap[coeff_names].to_numpy().flatten()

## Tests

### Test 1: manual tweaking of coefficients

Can we nudge the current best coefficients to find an even better set?

From the table above we can see that the best overall combo is not made up of the best combo for each deprivation quantile:

In [25]:
sum_sqres_cols = [c for c in df_best_gens.columns if c.startswith('sum_sqres')]

d1 = df_best_gens.sort('sum_sqres')[0][sum_sqres_cols]
d2 = df_best_gens[sum_sqres_cols].min()

d1 = d1.with_columns(pl.Series('coeff_combo', ['max_sum_sqres_overall']))
d2 = d2.with_columns(pl.Series('coeff_combo', [f'max_sum_sqres_each_depriv']))

display(pl.concat((d1, d2)))

sum_sqres,sum_sqres_q00,sum_sqres_q02,sum_sqres_q04,sum_sqres_q06,sum_sqres_q08,coeff_combo
f64,f64,f64,f64,f64,f64,str
236.659584,110.802377,108.182328,105.678815,103.340967,100.895037,"""max_sum_sqres_overall"""
236.659584,110.46065,107.82481,105.549064,103.167012,100.384223,"""max_sum_sqres_each_depriv"""


The second row of the table has the best sum of square residuals for each deprivation quantile. Its values are lower (better) for each quantile than in the overall best combo. (1s.f.: the second and the two most-deprived quantiles than in the overall best combo.)

View the best coefficients for these deprivation quantiles:

In [27]:
for q in ['q00', 'q02', 'q04', 'q06', 'q08']:
    # Columns for this depriv quantile:
    coeffs_q = [c for c in df_best_gens.columns if q in c]
    # Pick out coeffs where this sum_sqres is best for this quantile:
    mask = df_best_gens.sort('sum_sqres')[f'sum_sqres_{q}'] == df_best_gens[f'sum_sqres_{q}'].min()

    # Pick out the coeffs that give the best sum_sqres:
    d1 = df_best_gens.sort('sum_sqres')[0][coeffs_q]
    d2 = df_best_gens.sort('sum_sqres').filter(mask)[0][coeffs_q]
    # Add a new column to label where the values are from:
    d1 = d1.with_columns(pl.Series('coeff_combo', ['best_overall']))
    d2 = d2.with_columns(pl.Series('coeff_combo', [f'best_{q}']))

    display(pl.concat((d1, d2)))

age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,sum_sqres_q00,coeff_combo
f64,f64,f64,f64,f64,f64,str
0.00053,0.0034,0.0045,0.0078,0.014,110.802377,"""best_overall"""
0.00047,0.0036,0.005,0.0072,0.015,110.46065,"""best_q00"""


age_less65_q02,age_65_q02,age_70_q02,age_75_q02,age_over80_q02,sum_sqres_q02,coeff_combo
f64,f64,f64,f64,f64,f64,str
0.00045,0.0029,0.0041,0.006,0.013,108.182328,"""best_overall"""
0.00041,0.0026,0.004,0.006,0.014,107.82481,"""best_q02"""


age_less65_q04,age_65_q04,age_70_q04,age_75_q04,age_over80_q04,sum_sqres_q04,coeff_combo
f64,f64,f64,f64,f64,f64,str
0.00037,0.0026,0.0036,0.006,0.012,105.678815,"""best_overall"""
0.00036,0.0026,0.0037,0.006,0.012,105.549064,"""best_q04"""


age_less65_q06,age_65_q06,age_70_q06,age_75_q06,age_over80_q06,sum_sqres_q06,coeff_combo
f64,f64,f64,f64,f64,f64,str
0.00037,0.0024,0.0034,0.0054,0.011,103.340967,"""best_overall"""
0.00035,0.0024,0.0034,0.0054,0.011,103.167012,"""best_q06"""


age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08,sum_sqres_q08,coeff_combo
f64,f64,f64,f64,f64,f64,str
0.00032,0.0023,0.0034,0.0048,0.01,100.895037,"""best_overall"""
0.00033,0.002,0.003,0.0048,0.011,100.384223,"""best_q08"""


Manually adjust the values in the best combo to recreate the best sum of square residuals for each deprivation quantile.

In [29]:
# Pick out best coeffs:
mask = df_best_gens['sum_sqres'] == df_best_gens['sum_sqres'].min()
coeffs = df_best_gens.filter(mask)[coeff_names].to_numpy().flatten()
# Update some:
for q in ['q00', 'q02', 'q04', 'q06', 'q08']:
    # Columns for this depriv quantile:
    coeffs_q = [c for c in df_best_gens.columns if ((q in c) & ('age' in c))]
    # Pick out coeffs where this sum_sqres is best for this quantile:
    mask = df_best_gens.sort('sum_sqres')[f'sum_sqres_{q}'] == df_best_gens[f'sum_sqres_{q}'].min()
    # Pick out the coeffs that give the best sum_sqres:
    d2 = df_best_gens.sort('sum_sqres').filter(mask)[0][coeffs_q]
    # Update values:
    for c in coeffs_q:
        coeffs[coeff_names.index(c)] = d2[c].to_numpy().flatten()[0]
    
# coeffs[coeff_names.index('age_75_q02')] = 0.006
# coeffs[coeff_names.index('age_over80_q02')] = 0.014
# coeffs[coeff_names.index('age_75_q06')] = 0.006
# coeffs[coeff_names.index('age_over80_q08')] = 0.011

Check that coeffs still increase with age (across) and deprivation level (upwards):

In [30]:
coeffs.reshape(5, 5)

array([[0.00047, 0.0036 , 0.005  , 0.0072 , 0.015  ],
       [0.00041, 0.0026 , 0.004  , 0.006  , 0.014  ],
       [0.00036, 0.0026 , 0.0037 , 0.006  , 0.012  ],
       [0.00035, 0.0024 , 0.0034 , 0.0054 , 0.011  ],
       [0.00033, 0.002  , 0.003  , 0.0048 , 0.011  ]])

What's the effect on overall R^2?

In [31]:
sum_sqres, rat, fitness, ratios_by_age, sum_sqres_by_depriv = eval_admissions(
    coeffs,
    admissions_lists,
    x_lists,
    admissions_by_age
)

# Best R^2 from all genetic algorithm results:
fitness_best = df_best_gens['fitness'].min()
sum_sqres_best = df_best_gens['sum_sqres'].min()

print(f'New sum_sqres: {sum_sqres:.4f}')
print(f'Old sum_sqres: {sum_sqres_best:.4f}')

print(f'New fitness: {fitness:.4f}')
print(f'Old fitness: {fitness_best:.4f}')

New sum_sqres: 235.9847
Old sum_sqres: 236.6596
New fitness: 276.9253
Old fitness: 238.9292


The overall sum of square residuals has decreased with this manual changing of the coefficients - good - but the overall fitness taking into account numbers of admissions in each age band has increased - bad.

It is worth trying more combinations of coefficients near the found values to check whether any small adjustments can make better results. The genetic algorithm isn't likely to have tried every good combination by chance, and there could be other better options yet to be discovered.

### Test 2: Random offsets

Take the best set of coefficients. Nudge the 25 coefficients up or down a bit at random. Do any of the resulting combinations have better fitness than the starting coefficients?

Allow these offsets for the coeffs:

In [32]:
# offsets = np.array([1e-4, 1e-3, 1e-3, 1e-3, 1e-3] * 5)
# offsets = np.array([1e-5, 1e-4, 1e-4, 1e-4, 1e-3] * 5)
offsets = np.array([1e-5, 1e-4, 1e-4, 1e-4, 1e-4] * 5)

What probability of no change do we need to have an average of 3 coeffs changing?

In [33]:
((25.0 - 3.0)/25.0)

0.88

Generate 10,000 lists of changes to the starting parameters. Each change may be either zero, or plus or minus one or two steps from the start value. Each coefficient has an 88% chance of not changing, a 4% chance of moving up or down one step, and a 2% chance of moving up or down two steps.

In [34]:
seeds_list = []
random_offsets = []
for seed in np.arange(10000):
    np.random.seed(seed)
    r = list(offsets * np.random.choice(np.arange(-2, 3, 1), 25, p=[0.02, 0.04, 0.88, 0.04, 0.02]))
    if r not in random_offsets:
        random_offsets.append(r)
        seeds_list.append(seed)

Check how many sets of changes were kept after removing repeats:

In [35]:
len(random_offsets)

7730

For each set in turn, apply the changes to the set of starting parameters (here the best one of the 100 sets of outputs from the genetic algorithm). Check that the resulting coefficients obey the rules: they must increase with age band and with deprivation level. If this check is passed, then keep a copy of the resulting coefficients.

In [36]:
coeffs = coeffs_best_deap
# Start with a copy of the original coefficients and label them
# with seed -42. All other seeds are positive.
results = [list(coeffs)]
seeds_used = [-42]

for s, seed in enumerate(seeds_list):
    r = coeffs + random_offsets[s]
    # Check if rules are met - coeffs increase with age and depriv:
    r_arr = r.reshape(5, 5)
    age_increases = (np.diff(r_arr, axis=1) >= 0).all()
    depriv_increases = (np.diff(r_arr, axis=0) <= 0).all()

    # Store result if it meets the rules and hasn't previously
    # been stored:
    if age_increases & depriv_increases:
        if list(r) not in results:
            results.append(list(r))
            seeds_used.append(seed)

# Turn the results into a DataFrame:
df_nudged = pl.DataFrame(results, schema=coeff_names, orient='row')
df_nudged = df_nudged.insert_column(0, pl.Series('seed', seeds_used))

Check how many valid sets of coefficients there are:

In [37]:
len(df_nudged)

4498

View the first few sets of coefficients. Seed -42 is the starting set with no changes.

In [38]:
df_nudged.head()

seed,age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,age_less65_q02,age_65_q02,age_70_q02,age_75_q02,age_over80_q02,age_less65_q04,age_65_q04,age_70_q04,age_75_q04,age_over80_q04,age_less65_q06,age_65_q06,age_70_q06,age_75_q06,age_over80_q06,age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08
i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
-42,0.00049,0.0034,0.0052,0.0076,0.014,0.00045,0.0029,0.0041,0.0066,0.013,0.0004,0.0026,0.0035,0.0057,0.011,0.00035,0.0024,0.0034,0.0054,0.011,0.00035,0.0023,0.003,0.0048,0.01
2,0.00049,0.0033,0.0052,0.0076,0.014,0.00045,0.0029,0.0041,0.0066,0.013,0.0004,0.0026,0.0035,0.0057,0.011,0.00035,0.0024,0.0034,0.0054,0.011,0.00035,0.0023,0.003,0.0048,0.01
3,0.00049,0.0034,0.0052,0.0076,0.014,0.00045,0.0029,0.0041,0.0065,0.013,0.00039,0.0026,0.0035,0.0057,0.011,0.00035,0.0023,0.0034,0.0054,0.011,0.00035,0.0023,0.003,0.0048,0.01
6,0.00049,0.0034,0.0052,0.0075,0.014,0.00045,0.0029,0.0041,0.0066,0.013,0.0004,0.0026,0.0035,0.0057,0.011,0.00037,0.0024,0.0034,0.0054,0.011,0.00034,0.0023,0.003,0.0048,0.01
9,0.00047,0.0034,0.0052,0.0076,0.014,0.00045,0.0029,0.0041,0.0066,0.013,0.0004,0.0026,0.0036,0.0056,0.011,0.00035,0.0024,0.0034,0.0054,0.011,0.00035,0.0023,0.003,0.0048,0.011


Evaluate the fitnesses:

In [39]:
df_nudged = eval_fitness_to_df(df_nudged, admissions_lists, x_lists, admissions_by_age, coeff_names, concat=True)

Only keep the sets of coefficents where the fitness is at least as good as the starting value. How many are there, including the starting values?

In [40]:
fitness_orig = df_nudged.filter(df_nudged['seed'] == -42)['fitness'].to_numpy()[0]
mask = df_nudged['fitness'] <= fitness_orig

print(len(df_nudged.filter(mask)))

21


View these better coefficients' fitness values:

In [41]:
df_nudged.filter(mask).sort('fitness')[['seed', 'sum_sqres', 'wrong_rat', 'fitness']]

seed,sum_sqres,wrong_rat,fitness
i64,f64,f64,f64
8265,237.231563,1.005702,238.584192
2386,237.216011,1.006382,238.729991
2573,237.270058,1.0062,238.74115
1275,237.038793,1.007231,238.752727
7017,237.305855,1.006229,238.784139
…,…,…,…
3658,237.237449,1.007006,238.899439
533,237.319759,1.006672,238.903183
8761,237.317382,1.006731,238.914678


_Success!_ The best set of coefficients has both a better sum of square residuals and a better age-band wrongness ratio than the starting set (seed -42).

View the better sets of coefficients as a grid:

In [42]:
# Pick out original coefficients before any nudging:
coeffs_orig = df_nudged.filter(df_nudged['seed'] == -42)[coeff_names].to_numpy().flatten().reshape(5, 5)

# Only sets at least as good as starting set:
df = df_nudged.filter(mask).sort('fitness')
for i in range(len(df)):
    # Pick out fitness score and coefficients here:
    fitness_here = df_nudged.sort('fitness')[i]['fitness'].to_numpy().flatten()[0]
    coeffs_here = df_nudged.sort('fitness')[i][coeff_names].to_numpy().flatten().reshape(5, 5)
    # Display results:
    print(f'Fitness: {fitness_here:.2f}')
    print(coeffs_here)
    print('Change from starting coefficients:')
    print(coeffs_here - coeffs_orig)
    print('\n')

Fitness: 238.58
[[0.00049 0.0034  0.0052  0.0076  0.014  ]
 [0.00045 0.0029  0.004   0.0066  0.013  ]
 [0.0004  0.0026  0.0036  0.0057  0.011  ]
 [0.00035 0.0024  0.0034  0.0054  0.011  ]
 [0.00035 0.0023  0.003   0.0048  0.01   ]]
Change from starting coefficients:
[[ 0.e+00  0.e+00  0.e+00  0.e+00  0.e+00]
 [ 0.e+00  0.e+00 -1.e-04  0.e+00  0.e+00]
 [ 0.e+00  0.e+00  1.e-04  0.e+00  0.e+00]
 [ 0.e+00  0.e+00  0.e+00  0.e+00  0.e+00]
 [ 0.e+00  0.e+00  0.e+00  0.e+00  0.e+00]]


Fitness: 238.73
[[0.00049 0.0034  0.0052  0.0076  0.014  ]
 [0.00044 0.0029  0.0041  0.0066  0.013  ]
 [0.00041 0.0026  0.0035  0.0057  0.011  ]
 [0.00035 0.0024  0.0034  0.0054  0.011  ]
 [0.00035 0.0023  0.003   0.0048  0.01   ]]
Change from starting coefficients:
[[ 0.e+00  0.e+00  0.e+00  0.e+00  0.e+00]
 [-1.e-05  0.e+00  0.e+00  0.e+00  0.e+00]
 [ 1.e-05  0.e+00  0.e+00  0.e+00  0.e+00]
 [ 0.e+00  0.e+00  0.e+00  0.e+00  0.e+00]
 [ 0.e+00  0.e+00  0.e+00  0.e+00  0.e+00]]


Fitness: 238.74
[[0.00049 0.00

Now know in principle there are valid combinations near the starting values that have better fitness. The trick will be to check all sensible combinations of nudged coefficients to winkle out the best one.

Note that these good combinations mostly follow a pattern: there is a pair of changed coefficients that share a row or a column, and one of the pair has increased slightly and the other decreased slightly.

### Test 3: Optimise deprivation quantiles

Can we pick out a few combinations of coefficients that are good for each depriv quantile, then put the five quantiles together to get 25 coeffs that have good overall fitness?

Take the best coefficients. For each deprivation quantile, nudge the best coefficient one or two clicks either way and calculate the new fitness scores.

__Calculate new options and fitnesses__

The following function makes every combination of the five (not 25) start coefficients nudged up or down one or two steps:

In [43]:
def generate_new_combos_depriv(coeffs_this_depriv):
    # offsets = np.array([1e-4, 1e-3, 1e-3, 1e-3, 1e-3])
    # offsets = np.array([1e-5, 1e-4, 1e-4, 1e-4, 1e-3])
    offsets = np.array([1e-5, 1e-4, 1e-4, 1e-4, 1e-4])
    
    # Add on the offsets to make the new options:
    # Set up each value to be nudged one or two sig figs up or down.
    new_options = [
        [round(coeffs_this_depriv[j] + i*offsets[j], 5) for i in range(-2, 3)] for j in range(5)
    ]
    # Generate all combinations of these new parameters.
    # Each element of this list is a tuple of five new coeffs.
    all_new_option_combos = list(itertools.product(*new_options))
    # Only keep combinations where coefficients increase with age band:
    mask = (np.diff(all_new_option_combos, axis=1) >= 0).all(axis=1)
    all_new_option_combos = np.array(all_new_option_combos)[mask]
    return all_new_option_combos

Use the best coefficients from the genetic algorithm output:

In [44]:
start_coeffs = coeffs_best_deap

start_coeffs.reshape(5, 5)

array([[0.00049, 0.0034 , 0.0052 , 0.0076 , 0.014  ],
       [0.00045, 0.0029 , 0.0041 , 0.0066 , 0.013  ],
       [0.0004 , 0.0026 , 0.0035 , 0.0057 , 0.011  ],
       [0.00035, 0.0024 , 0.0034 , 0.0054 , 0.011  ],
       [0.00035, 0.0023 , 0.003  , 0.0048 , 0.01   ]])

Find every nudged variation on the best coefficients and calculate the new r-squared for that deprivation quantile.

In [45]:
new_combo_df_dict = {}

for quantile_str in quantile_str_list:
    # Set up final column names:
    depriv_ind = quantile_str_list.index(quantile_str)
    best_coeff_cols = [c for c in coeff_names if quantile_str in c]
    col_sqres = f'sum_sqres_{quantile_str}'

    # Find all nudged coefficients for this deprivation band:
    coeffs_this_depriv = start_coeffs[depriv_ind*5:depriv_ind*5+5]
    all_new_option_combos = generate_new_combos_depriv(coeffs_this_depriv)

    # Calculate fitnesses of each of the nudged sets.
    # Store results in here:
    list_sum_sqres = []
    for c, coeffs in enumerate(all_new_option_combos):
        # Population numbers for areas in this quantile:
        # Separate prediction for each deprivation quantile:
        yhat = predict_admissions(x_lists[depriv_ind], np.array(list(coeffs)))
        # Compare with the observed admissions to calculate sum of square residuals:
        sum_sqres = find_square_residuals(yhat, admissions_lists[depriv_ind])
        # Store result:
        list_sum_sqres.append(sum_sqres)

    # Place new coeff combos into dataframe:
    df_new_combos = pl.DataFrame(all_new_option_combos, schema=best_coeff_cols, orient='row')
    # Make a new column with the sum of square residuals values:
    df_new_combos = df_new_combos.with_columns(pl.Series(col_sqres, list_sum_sqres))
    
    new_combo_df_dict[quantile_str] = df_new_combos

View the results:

In [46]:
for quantile_str, df in new_combo_df_dict.items():
    print(quantile_str)
    # Print the best coefficients for comparison:
    col_sqres = f'sum_sqres_{quantile_str}'
    cols = [c for c in coeff_names if quantile_str in c] + [col_sqres, 'sum_sqres']
    print('Coeffs in best overall combo:')
    display(df_coeffs_best_deap[cols])
    best_sqres = df_coeffs_best_deap[col_sqres]
    # How many found values are better than the starting set?
    n_better = len(df.filter(df[col_sqres] <= best_sqres))
    # Print the nudged coeffs from best to worst:
    print('Nudged coeffs:')
    print(f'{n_better} options out of {len(df)} total are as good or better than start coeffs.')
    print(f'(display best options out of {len(df)} total)')
    display(df.sort(col_sqres).head())
    print('\n'*2)

q00
Coeffs in best overall combo:


age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,sum_sqres_q00,sum_sqres
f64,f64,f64,f64,f64,f64,f64
0.00049,0.0034,0.0052,0.0076,0.014,110.556611,237.338713


Nudged coeffs:
176 options out of 3125 total are as good or better than start coeffs.
(display best options out of 3125 total)


age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,sum_sqres_q00
f64,f64,f64,f64,f64,f64
0.00048,0.0036,0.0054,0.0075,0.014,110.481803
0.00048,0.0036,0.0054,0.0074,0.014,110.482564
0.00048,0.0036,0.0054,0.0076,0.014,110.487148
0.00047,0.0034,0.005,0.0074,0.015,110.48968
0.00047,0.0035,0.005,0.0074,0.015,110.49006





q02
Coeffs in best overall combo:


...

age_less65_q06,age_65_q06,age_70_q06,age_75_q06,age_over80_q06,sum_sqres_q06
f64,f64,f64,f64,f64,f64
0.00033,0.0022,0.0032,0.0052,0.012,102.733323
0.00033,0.0022,0.0032,0.0053,0.012,102.782489
0.00033,0.0023,0.0032,0.0052,0.012,102.813478
0.00033,0.0022,0.0033,0.0052,0.012,102.815969
0.00033,0.0022,0.0032,0.0054,0.012,102.849191





q08
Coeffs in best overall combo:


age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08,sum_sqres_q08,sum_sqres
f64,f64,f64,f64,f64,f64,f64
0.00035,0.0023,0.003,0.0048,0.01,101.353211,237.338713


Nudged coeffs:
436 options out of 3125 total are as good or better than start coeffs.
(display best options out of 3125 total)


age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08,sum_sqres_q08
f64,f64,f64,f64,f64,f64
0.00033,0.0021,0.0028,0.0047,0.011,100.309782
0.00033,0.0021,0.0028,0.0046,0.011,100.310685
0.00033,0.0021,0.0029,0.0046,0.011,100.32486
0.00033,0.0021,0.0028,0.0048,0.011,100.327053
0.00033,0.0022,0.0028,0.0046,0.011,100.33596


For all deprivation quantiles, the sets of nudged coeffs contain an option with lower sum of square residuals than the one in the best overall combo.

__Gather five best sets of results__

Gather the coefficients that are best for each deprviation quantile:

In [47]:
best_combos = []
best_combo_cols = []

for q, quantile_str in enumerate(list(new_combo_df_dict.keys())):
    df = new_combo_df_dict[quantile_str].sort(f'sum_sqres_{quantile_str}')
    df = df[0] #  if q < 4 else df[3]
    cols = [c for c in df.columns if c.startswith('age')]
    best_combo_cols += cols
    best_combos += list(df[cols].to_numpy().flatten())

Check that the complete set of coefficients meet the requirements of increasing with age and deprivation:

In [48]:
print(np.array(best_combos).reshape(5, 5))

print('\nCheck coeffs increase with age band (across):')
print(np.diff(np.array(best_combos).reshape(5, 5), axis=1) >= 0)

print('\nCheck coeffs increase with deprivation (downwards):')
print(np.diff(np.array(best_combos).reshape(5, 5), axis=0) <= 0)

[[0.00048 0.0036  0.0054  0.0075  0.014  ]
 [0.00043 0.0029  0.0041  0.0064  0.013  ]
 [0.00038 0.0024  0.0034  0.0059  0.012  ]
 [0.00033 0.0022  0.0032  0.0052  0.012  ]
 [0.00033 0.0021  0.0028  0.0047  0.011  ]]

Check coeffs increase with age band (across):
[[ True  True  True  True]
 [ True  True  True  True]
 [ True  True  True  True]
 [ True  True  True  True]
 [ True  True  True  True]]

Check coeffs increase with deprivation (downwards):
[[ True  True  True  True  True]
 [ True  True  True  True  True]
 [ True  True  True  True  True]
 [ True  True  True  True  True]]


These requirements are met!

__Pick best valid combination of good options__

Instead, try a few combinations of "best" results. For each deprivation quantile in turn, use its best results and then select results for the other quantiles that work around it.

The following functions search an array of coefficients to find sets that work with a given set of coefficients. They find either coefficients that always increase or stay the same as the reference set, or that always decrease or stay the same.

In [48]:
def pick_conditions_met(combo, all_combos, pick_more=True):
    """
    Find options where all coeffs are more than/same as fixed values.

    Inputs
    ------
    combo      - list. Fixed coefficients.
    all_combos - np.array. One row per set of coefficients.
                 Assume sorted from best to worst r-squared.
    Returns
    -------
    list. The selected coefficients from the big list.
    """
    # List of masks, one mask per age band.
    # Masks are for whether the coeffs for this age band
    # are more or equal to the fixed value for this age band.
    if pick_more:
        masks = [(all_combos[:, p] >= combo[p]) for p in range(len(combo))]
    else:
        masks = [(all_combos[:, p] <= combo[p]) for p in range(len(combo))]
    # Gather masks:
    masks = np.vstack(masks).T
    # Check where condition is met for all age bands:
    valid_mask = masks.all(axis=1)
    # Select this index "pick", the first in the list where the
    # condition is met for all age bands.
    pick = np.where(valid_mask == True)[0][0]
    # Return a list of the coefficients in this row of the table:
    return all_combos[pick]

Pick out combos:

In [49]:
def pick_combos(fixed_combo, new_combo_100_df_dict, q):
    """
    Pick combos of all good coeffs that meet rules.

    Fix the coefficients for the qth quantile in the list.
    Then pick coefficients for the adjacent quantiles that meet rules
    and continue until all quantiles have been picked.
    Picked coefficients must increase with age band and with level
    of deprivation.

    Inputs
    ------
    fixed_combo           - list. Five coefficients that are fixed.
    new_combo_100_df_dict - dict. One entry per deprivation quantile.
                            Each entry is a dataframe of coefficients
                            and their r-squared value.
    q                     - int. Index of depriv quantile for the
                            fixed combo.

    Returns
    -------
    best_combo - list. Length 25, picked coefficients.
    """
    def gather_coeff_combos(p):
        """
        Change big dataframe of coeff combos to array for picking.
        """
        # Find the name of the r-squared column for the next combo:
        p_str = list(new_combo_100_df_dict.keys())[p]
        # Pick out the coefficients and r-squared for this combo:
        arr = new_combo_100_df_dict[p_str].sort(f'sum_sqres_{p_str}').to_numpy()
        # Cut off r-squared column:
        arr = arr[:, :-1]
        return arr
        
    # Set up coeff storage.
    # Store the coeffs for each quantile in their own list in order
    # in the following list of lists. Start with empty lists...
    best_combo = [[]] * 5
    # ... fill in one list of coeffs...
    best_combo[q] = list(fixed_combo)
    # ... then use that filled list to select the adjacent lists.
    for p in range(q+1, 5):
        # Find the combos where values should be less than or same as the
        # starting values.
        arr = gather_coeff_combos(p)
        # Find the best coeff combo that meets the decreasing criteria:
        picked_coeffs = pick_conditions_met(best_combo[p-1], arr, pick_more=False)
        # Store result:
        best_combo[p] = list(picked_coeffs)
    for p in range(q-1, -1, -1):
        # Find the combos where values should be more than or same as the
        # starting values.
        arr = gather_coeff_combos(p)
        # Find the best coeff combo that meets the increasing criteria:
        picked_coeffs = pick_conditions_met(best_combo[p+1], arr)
        # Store result:
        best_combo[p] = list(picked_coeffs)
    # Return a single flat list of coeffs, length 25.
    best_combo = sum(best_combo, [])
    return best_combo

Run this function for each quantile in turn. For each loop, a different quantile's coefficients are fixed first. Then the other quantiles' coefficients have to work around those fixed values.

In [50]:
best_picked_combos = {}
# Each item will be a list of 25 coeffs.

for q, quantile_str in enumerate(list(new_combo_df_dict.keys())):
    # Pick out a list of five coefficients for this depriv quantile:
    fixed_combo = (new_combo_df_dict[quantile_str]
                   .sort(f'sum_sqres_{quantile_str}')[0]
                   .to_numpy().flatten()[:-1])
    # Pick out coeffs for the other quantiles that work well with this:
    best_picked_combos[quantile_str] = pick_combos(
        fixed_combo, new_combo_df_dict, q)

Calculate their sum of square residuals:

In [51]:
best_picked_combos_r2s = {}

for quantile_str, best_combo in best_picked_combos.items():
    # Calculate fitnesses of each of the nudged sets.
    sum_sqres, sum_sqres_by_depriv = many_sum_sqres(best_combo, x_lists, admissions_lists)
    best_picked_combos_r2s[quantile_str] = sum_sqres_by_depriv + [sum_sqres]

Rearrange the picked coefficients into dataframes:

In [52]:
best_picked_combos_dfs = {}
age_label_list = ['less65', '65-70', '70-75', '75-80', 'over80']

for quantile_str, best_combo in best_picked_combos.items():
    # Make a dataframe:
    data = np.array(best_combo).reshape(5, 5)
    # Add deprivation column:
    data = np.hstack((np.array(quantile_str_list).reshape(5, 1), data))
    df = pl.DataFrame(data, schema=['depriv_quantile_min'] + age_label_list)
    for col in age_label_list:
        df = df.with_columns(pl.col(col).cast(float))
    # Place r-squared results in the dataframe:
    list_sum_sqres = best_picked_combos_r2s[quantile_str]
    df = df.with_columns(pl.Series('sum_sqres', list_sum_sqres[:-1]))
    
    best_picked_combos_dfs[quantile_str] = df

View the results:

In [53]:
for quantile_str, df in best_picked_combos_dfs.items():
    print(f'First fixed: {quantile_str}')
    print(f'Overall sum sqres: {best_picked_combos_r2s[quantile_str][-1]:.5f}')
    display(df)
    print('')

First fixed: q00
Overall sum sqres: 234.50419


depriv_quantile_min,less65,65-70,70-75,75-80,over80,sum_sqres
str,f64,f64,f64,f64,f64,f64
"""q00""",0.0004,0.006,0.006,0.006,0.014,110.354891
"""q02""",0.0004,0.003,0.004,0.006,0.014,107.800573
"""q04""",0.0003,0.002,0.003,0.006,0.014,104.55857
"""q06""",0.0003,0.002,0.003,0.006,0.012,102.55868
"""q08""",0.0002,0.002,0.003,0.005,0.012,98.702951



First fixed: q02
Overall sum sqres: 234.50419


depriv_quantile_min,less65,65-70,70-75,75-80,over80,sum_sqres
str,f64,f64,f64,f64,f64,f64
"""q00""",0.0004,0.006,0.006,0.006,0.014,110.354891
"""q02""",0.0004,0.003,0.004,0.006,0.014,107.800573
"""q04""",0.0003,0.002,0.003,0.006,0.014,104.55857
"""q06""",0.0003,0.002,0.003,0.006,0.012,102.55868
"""q08""",0.0002,0.002,0.003,0.005,0.012,98.702951



First fixed: q04
Overall sum sqres: 234.63866


depriv_quantile_min,less65,65-70,70-75,75-80,over80,sum_sqres
str,f64,f64,f64,f64,f64,f64
"""q00""",0.0004,0.005,0.005,0.007,0.015,110.442113
"""q02""",0.0004,0.003,0.003,0.007,0.014,107.823731
"""q04""",0.0003,0.002,0.002,0.007,0.014,104.525757
"""q06""",0.0003,0.002,0.002,0.007,0.012,102.623322
"""q08""",0.0002,0.002,0.002,0.007,0.012,98.86709



First fixed: q06
Overall sum sqres: 234.50419


depriv_quantile_min,less65,65-70,70-75,75-80,over80,sum_sqres
str,f64,f64,f64,f64,f64,f64
"""q00""",0.0004,0.006,0.006,0.006,0.014,110.354891
"""q02""",0.0004,0.003,0.004,0.006,0.014,107.800573
"""q04""",0.0003,0.002,0.003,0.006,0.014,104.55857
"""q06""",0.0003,0.002,0.003,0.006,0.012,102.55868
"""q08""",0.0002,0.002,0.003,0.005,0.012,98.702951



First fixed: q08
Overall sum sqres: 234.73615


depriv_quantile_min,less65,65-70,70-75,75-80,over80,sum_sqres
str,f64,f64,f64,f64,f64,f64
"""q00""",0.0004,0.006,0.006,0.006,0.014,110.354891
"""q02""",0.0004,0.003,0.004,0.006,0.014,107.800573
"""q04""",0.0002,0.003,0.004,0.005,0.014,104.839407
"""q06""",0.0002,0.003,0.004,0.005,0.012,102.870443
"""q08""",0.0001,0.003,0.004,0.004,0.012,98.631977


Compare the overall r-squared scores for the five options:

In [54]:
pl.DataFrame(
    np.vstack((
        quantile_str_list,
        [round(best_picked_combos_r2s[quantile_str][-1], 3) for quantile_str in quantile_str_list]
    )).T,
    schema=['fixed_depriv_quantile_min', 'sum_sqres']
)

fixed_depriv_quantile_min,sum_sqres
str,str
"""q00""","""234.504"""
"""q02""","""234.504"""
"""q04""","""234.639"""
"""q06""","""234.504"""
"""q08""","""234.736"""


The lowest overall sum of square residuals is for the sets of coefficients that first fixed the 0.0-0.2, 0.2-0.4 and 0.6-0.8 deprivation quantiles.

Compare results with SSNAP-derived coefficients

Recalculate total fitness:

In [55]:
sum_sqres, rat, fitness, ratios_by_age, sum_sqres_by_depriv = eval_admissions(
    best_picked_combos['q00'],
    admissions_lists,
    x_lists,
    admissions_by_age
)

# Best R^2 from all genetic algorithm results:
fitness_best = df_best_gens['fitness'].min()
sum_sqres_best = df_best_gens['sum_sqres'].min()

print(f'New sum_sqres: {sum_sqres:.4f}')
print(f'Old sum_sqres: {sum_sqres_best:.4f}')

print(f'New fitness: {fitness:.4f}')
print(f'Old fitness: {fitness_best:.4f}')

New sum_sqres: 234.5042
Old sum_sqres: 237.4549
New fitness: 338.3078
Old fitness: 252.5461


Sum of square residuals has improved using these nudged coefficients, but fitness is much worse.

So it doesn't look likely that we'll find a good fit by looking at deprivation band alone.

### Test 4: Optimise age bands

Can we pick out a few combinations of coefficients that are good for each age band, then put five sets together to get 25 coeffs that have good overall fitness?

The randomly-picked best fits seem to improve the "wrongness" ratio in the age bands much more than the overall sum of square residuals. So find ways to nudge the coefficients that result in better fits to the total national admissions for each age band.

__Calculate new options and fitnesses__

In [49]:
dict_offsets_for_ages = {  # np.array([1e-4, 1e-3, 1e-3, 1e-3, 1e-3])
#     'less65': 1e-4,
#     '65': 1e-3,
#     '70': 1e-3,
#     '75': 1e-3,
#     'over80': 1e-3,
# }   
    'less65': 1e-5,
    '65': 1e-4,
    '70': 1e-4,
    '75': 1e-4,
    # 'over80': 1e-3,
    'over80': 1e-4,
}

All combos of offsets for ages:

In [50]:
combo_scales_for_ages = np.array([c for c in itertools.product([-2, -1, 0, 1, 2], repeat=5)])

In [51]:
len(combo_scales_for_ages)

3125

For a set of coeffs, apply these offsets and then recalculate fitness:

In [52]:
def wrong_rat(x_lists_here, coeffs, admissions_by_age):
    # Predictions for each age band across England:
    # Separate prediction for each deprivation quantile:
    # coeffs = individual[(i*5):(i*5)+5] #* np.array(coeffs_ssnap)
    yhat_list = (
        [(x_lists_here[i] * coeffs[i]).sum() for i in range(len(coeffs))]
    )
    predictions_by_age = sum(yhat_list)
    # Divide the admissions by age band by the observed values:
    predictions_by_age /= admissions_by_age_here
    return predictions_by_age

Pick out best coefficients again and keep the fitness columns:

In [53]:
start_coeffs = df_best_gens.sort('fitness')[0]

In [54]:
age_bands = ['less65', '65', '70', '75', 'over80']

dict_nudged_dfs = {}
for a, age_band in enumerate(age_bands):
    x_lists_here = [x_lists[i][a] for i in range(5)]  # only data for this age band
    admissions_by_age_here = admissions_by_age[a]
    
    age_cols = [c for c in df_best_gens.columns if f'_{age_band}_' in c]
    age_offset = dict_offsets_for_ages[age_band]
    combo_offsets_for_age = combo_scales_for_ages * age_offset
    
    start_coeffs_here = start_coeffs[age_cols].to_numpy().flatten()

    df_nudge_here = pl.DataFrame()    
    for c, combo in enumerate(combo_offsets_for_age):
        coeffs_here = start_coeffs_here + np.array(combo)
        # Check that coeffs increase with age band:
        if all(np.diff(coeffs_here) <= 0.0) & all(coeffs_here > 0.0):
            rat = wrong_rat(x_lists_here, coeffs_here, admissions_by_age_here)
            abs_rat = abs(1.0 - rat)
            row = dict(zip(age_cols + ['rat', 'abs_rat'], list(coeffs_here) + [round(rat, 7), round(abs_rat, 7)]))
            df_nudge_here = pl.concat((df_nudge_here, pl.DataFrame(row)))
        else:
            # Don't calculate results for here.
            pass
    # Store result:
    dict_nudged_dfs[age_band] = df_nudge_here

__Check where middle quartile matches SSNAP-derived coefficients__

To reduce the number of options somewhat, only check the results where the middle deprivation quantile has the same value as the SSNAP-derived coefficient.

Pick out what these values are when rounded to the same precision as the results:

In [55]:
coeffs_ssnap

array([0.000408, 0.002644, 0.003735, 0.006005, 0.011558])

In [56]:
coeffs_ssnap_round = [0.00041, 0.0026, 0.0037, 0.0060, 0.0116]

Pick out just results where the values match:

In [57]:
i = 0
dict_nudged_dfs_ssnapq04 = {}
for key, df in dict_nudged_dfs.items():
    dict_nudged_dfs_ssnapq04[key] = df.filter(df[f'age_{key}_q04'] == coeffs_ssnap_round[i]).sort('abs_rat')
    i += 1

__Pick best valid combination of good options__

Try a different combination method to the deprivation test. Here, try all combinations of the top six sets for each age band. Maybe the combination of the 5th best for each makes the 1st best overall.
The top 6 has been picked because it runs a decent amount (7776 combos) but doesn't take forever to run.

In [58]:
# Every combo of the top six indices:
index_combos = [c for c in itertools.product(range(6), repeat=5)]

In [60]:
all_coeff_cols

['age_less65_q00',
 'age_less65_q02',
 'age_less65_q04',
 'age_less65_q06',
 'age_less65_q08',
 'age_65_q00',
 'age_65_q02',
 'age_65_q04',
 'age_65_q06',
 'age_65_q08',
 'age_70_q00',
 'age_70_q02',
 'age_70_q04',
 'age_70_q06',
 'age_70_q08',
 'age_75_q00',
 'age_75_q02',
 'age_75_q04',
 'age_75_q06',
 'age_75_q08',
 'age_over80_q00',
 'age_over80_q02',
 'age_over80_q04',
 'age_over80_q06',
 'age_over80_q08']

In [61]:
all_coeffs

[0.0034,
 0.0029,
 0.0026,
 0.0026,
 0.0021,
 0.0050999999999999995,
 0.0041,
 0.0037,
 0.0031999999999999997,
 0.0031,
 0.015,
 0.013999999999999999,
 0.012,
 0.011,
 0.008]

In [63]:
fitness_keys = [
    'sum_sqres',
    'wrong_rat',
    'fitness',
    'wrong_rat_under65',
    'wrong_rat_65',
    'wrong_rat_70',
    'wrong_rat_75',
    'wrong_rat_over80',
    'sum_sqres_q00',
    'sum_sqres_q02',
    'sum_sqres_q04',
    'sum_sqres_q06',
    'sum_sqres_q08',
]

df_index_combos = pl.DataFrame()

for i, index_combo in enumerate(index_combos):
    print(f'{i+1:4d} out of {len(index_combos)}', end='\r')
    
    dict_inds = {
        'less65': index_combo[0],
        '65': index_combo[1],
        '70': index_combo[2],
        '75': index_combo[3],
        'over80': index_combo[4]
    }
    
    # Gather coeffs for these inds:
    all_coeff_cols = []
    all_coeffs = []
    for age_band, df in dict_nudged_dfs.items():  # dict_nudged_dfs_ssnapq04
        # 'less65', '65', '70', '75', 'over80'
        # Pick out the coeffs for this age band:
        coeff_cols_here = [c for c in df.columns if c.startswith('age')]
        all_coeff_cols += coeff_cols_here
        all_coeffs += list(df[dict_inds[age_band]][coeff_cols_here].to_numpy().flatten())
    # Order coeffs as usual:
    df_coeffs = pl.DataFrame(np.array(all_coeffs).reshape(1, len(all_coeffs)), schema=all_coeff_cols)
    df_coeffs = df_coeffs[coeff_names]
    coeffs = df_coeffs.to_numpy().flatten().reshape(5, 5)
    # print(coeffs)
    # print('\n')

    # Check that coeffs increase with deprivation:
    if np.all(np.diff(coeffs, axis=1) >= 0.0):
        # Calculate fitness:
        sum_sqres, rat, fitness, ratios_by_age, sum_sqres_by_depriv = eval_admissions(
            coeffs.flatten(),
            admissions_lists,
            x_lists,
            admissions_by_age
        )
        # Gather values for results:
        results_values = np.concatenate((coeffs.flatten(), [sum_sqres, rat, fitness], ratios_by_age, sum_sqres_by_depriv))
        df_coeffs = pl.DataFrame(results_values.reshape(1, len(results_values)), schema=coeff_names + fitness_keys)
        # Store result:
        df_index_combos = pl.concat((df_index_combos, df_coeffs))
    else:
        # Don't bother with coeffs that don't increase with deprivation.
        pass

__View results__

In [64]:
df_index_combos.sort('fitness')

age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,age_less65_q02,age_65_q02,age_70_q02,age_75_q02,age_over80_q02,age_less65_q04,age_65_q04,age_70_q04,age_75_q04,age_over80_q04,age_less65_q06,age_65_q06,age_70_q06,age_75_q06,age_over80_q06,age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08,sum_sqres,wrong_rat,fitness,wrong_rat_under65,wrong_rat_65,wrong_rat_70,wrong_rat_75,wrong_rat_over80,sum_sqres_q00,sum_sqres_q02,sum_sqres_q04,sum_sqres_q06,sum_sqres_q08
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.00047,0.0032,0.005,0.0074,0.012,0.00043,0.0027,0.0039,0.0064,0.011,0.00038,0.0024,0.0033,0.0055,0.01,0.00035,0.0024,0.0032,0.0052,0.01,0.00035,0.0022,0.0031,0.005,0.009,248.862521,1.251659,311.491057,0.970661,0.94973,0.963913,0.98134,0.882696,116.626738,113.652502,114.555367,106.890558,104.236958
0.00047,0.0032,0.005,0.0074,0.012,0.00043,0.0027,0.0039,0.0064,0.011,0.00038,0.0024,0.0033,0.0055,0.01,0.00035,0.0023,0.0032,0.0052,0.01,0.00035,0.0023,0.0031,0.005,0.009,248.90944,1.251888,311.60664,0.970661,0.949502,0.963913,0.98134,0.882696,116.626738,113.652502,114.555367,107.310853,103.916652
0.00047,0.0032,0.005,0.0074,0.012,0.00043,0.0027,0.0039,0.0064,0.011,0.00038,0.0024,0.0033,0.0055,0.011,0.00035,0.0023,0.0032,0.0052,0.01,0.00035,0.0023,0.0031,0.005,0.008,249.091407,1.254682,312.530394,0.970661,0.949502,0.963913,0.98134,0.879902,116.626738,113.652502,108.960336,107.310853,110.181069
0.00047,0.0032,0.005,0.0074,0.012,0.00043,0.0027,0.0039,0.0064,0.011,0.00038,0.0024,0.0033,0.0055,0.011,0.00035,0.0024,0.0032,0.0052,0.01,0.00035,0.0022,0.0031,0.005,0.008,249.173954,1.254453,312.577048,0.970661,0.94973,0.963913,0.98134,0.879902,116.626738,113.652502,108.960336,106.890558,110.774663
0.00047,0.0032,0.005,0.0074,0.012,0.00043,0.0027,0.0039,0.0064,0.011,0.00038,0.0024,0.0033,0.0055,0.01,0.00035,0.0024,0.0032,0.0052,0.01,0.00035,0.0022,0.0031,0.0049,0.009,248.989599,1.255711,312.658967,0.970661,0.94973,0.963913,0.977288,0.882696,116.626738,113.652502,114.555367,106.890558,104.539991
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
0.00047,0.0032,0.005,0.0074,0.012,0.00043,0.0027,0.0039,0.0064,0.011,0.00038,0.0024,0.0033,0.0055,0.009,0.00033,0.0022,0.0033,0.0052,0.009,0.00033,0.0021,0.0028,0.0046,0.008,262.388996,1.382331,362.70832,0.952447,0.924975,0.951476,0.965132,0.823639,116.626738,113.652502,122.399985,115.901059,117.959681
0.00047,0.0032,0.005,0.0074,0.012,0.00043,0.0027,0.0039,0.0064,0.011,0.00038,0.0024,0.0033,0.0055,0.009,0.00034,0.0022,0.0032,0.0052,0.009,0.00033,0.0021,0.0028,0.0046,0.008,262.329709,1.384046,363.076445,0.956994,0.924975,0.945213,0.965132,0.823639,116.626738,113.652502,122.399985,115.766776,117.959681
0.00047,0.0032,0.005,0.0074,0.012,0.00043,0.0027,0.0039,0.0064,0.011,0.00038,0.0024,0.0033,0.0055,0.009,0.00033,0.0022,0.0032,0.0052,0.009,0.00033,0.0021,0.0028,0.0047,0.008,262.449856,1.384542,363.372756,0.952447,0.924975,0.945213,0.969184,0.823639,116.626738,113.652502,122.399985,116.717516,117.288128


Starting values:

In [65]:
start_coeffs[coeff_names].to_numpy().reshape(5, 5)

array([[0.00049, 0.0034 , 0.0052 , 0.0076 , 0.014  ],
       [0.00045, 0.0029 , 0.0041 , 0.0066 , 0.013  ],
       [0.0004 , 0.0026 , 0.0035 , 0.0057 , 0.011  ],
       [0.00035, 0.0024 , 0.0034 , 0.0054 , 0.011  ],
       [0.00035, 0.0023 , 0.003  , 0.0048 , 0.01   ]])

Best combo from age bands alone:

In [66]:
df_index_combos.sort('fitness')[0][coeff_names].to_numpy().reshape(5, 5)

array([[0.00047, 0.0032 , 0.005  , 0.0074 , 0.012  ],
       [0.00043, 0.0027 , 0.0039 , 0.0064 , 0.011  ],
       [0.00038, 0.0024 , 0.0033 , 0.0055 , 0.01   ],
       [0.00035, 0.0024 , 0.0032 , 0.0052 , 0.01   ],
       [0.00035, 0.0022 , 0.0031 , 0.005  , 0.009  ]])

Best combo for age bands has moved more weight to the combo of highest deprivation and lowest age band.

Fitness from start values:

In [67]:
start_coeffs['fitness']

fitness
f64
238.929181


Fitnesses of these best values:

In [68]:
df_index_combos.sort('fitness')[['sum_sqres', 'wrong_rat', 'fitness']].head()

sum_sqres,wrong_rat,fitness
f64,f64,f64
248.862521,1.251659,311.491057
248.90944,1.251888,311.60664
249.091407,1.254682,312.530394
249.173954,1.254453,312.577048
248.989599,1.255711,312.658967


Best fitnesses worse than the start value! So can't find good coefficients from just the age bands alone.

## Conclusion

The tests show that to improve the overall fitness, we will need to calculate the coefficients for both the age bands and the deprivation bands simultaneously.